# 30 — Per-donor pert2state: patient-specific regulators of the MF malignant-CD4 program

**Extends nb29.** nb29 fitted one cohort signature (malignant vs reactive CD4, 66 donors) on the GWCD4i
Stim48hr knockdown matrix and got held-out r = 0.61. Here one signature is fitted **per donor** (30 donors
with ≥ 600 malignant and ≥ 600 reactive CD4 cells) and we ask: which regulators recur across patients, how
patients group by their regulator weights, and what the top recurrent regulators control downstream.

**Key simplification — X is identical for every donor.** The Perturb-seq matrix (genes × 2,582 regulators,
on-target entry masked), its column standardization and the TruncatedSVD (D = 60) are computed **once**;
only `y` changes. Each donor is one elastic net with **frozen** `alpha` / `l1_ratio` (chosen once on the
30-donor cohort signature with the authors' inner 4-fold CV) on the same 60-dim projection and the **same**
gene folds (5-fold × 3 repeats), so all weights live in one coordinate system and are directly comparable.

**Per-donor signature.** Each donor's malignant and reactive cells are split into 3 seeded, balanced
pseudo-replicates each → 6 pseudobulks → pydeseq2 `~ arm` → Wald `stat`. The pseudo-replicate variance is
sampling-only so |z| is inflated; `prepare_y` standardizes each signature, so only the per-gene relative
weighting matters.

**Sign convention (nb29).** $w_k > 0$: knocking down $k$ pushes a normal CD4 cell toward the malignant
program; $w_k < 0$: the program depends on $k$.

**Deviations from the authors' code path, all deliberate:** scaler + SVD fitted once on all genes instead of
inside every fold (unsupervised, so held-out r is only marginally optimistic); hyper-parameters frozen
across donors instead of re-tuned per fit (weights are only comparable if shrinkage is identical);
LIC x-axis = average linkage on correlation distance (nb29: Ward).

**Run order.** nb29 (§0, §2 caches) → `bash jobs/run_pseq30.sh` (signatures, design, 61 fits; ~1 h CPU) →
this notebook (light: reads the caches, every cell runs on the login kernel in minutes).

In [ ]:
# ============================================================
# §0  Parameters, imports, style
# ============================================================
import os, sys, importlib, warnings
from pathlib import Path
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data from {start}")


NB_DIR = _resolve_nb_dir(); print("NB_DIR =", NB_DIR)
sys.path.insert(0, str(NB_DIR / "helpers"))
import numpy as np, pandas as pd, scipy.stats as ss
import matplotlib.pyplot as plt, matplotlib as mpl
import scipy.cluster.hierarchy as sch
import pseq30_helpers as H
import perturbseq_helpers as P
import tcr_signaling_helpers as TH
for _m in (H, P, TH):
    importlib.reload(_m)
warnings.filterwarnings("ignore", category=FutureWarning)
H.set_style()
TEAL, DARK, MUTED, GREY, SCR = (H.DESIGN[k] for k in ("teal", "dark", "muted", "grey", "scr"))
SEED = H.SEED
np.random.seed(SEED)

missing = [p.name for p in (H.DONOR_TAB, H.GENE_SET, H.DONOR_Z, H.COHORT_DE, H.SVD_NPZ, H.LIC_ORDER,
                            H.HYPER, H.REACT_COMP, H.MODEL_FIT, H.WEIGHTS, H.W_SEM) if not p.exists()]
assert not missing, f"missing caches {missing} -- run `bash jobs/run_pseq30.sh` first"
print("knobs:", {k: getattr(H, k) for k in ("MIN_CELLS_ARM", "N_PSEUDO", "GENE_FRAC_DONORS", "MIN_CELLS_GENE",
                                             "N_PCS", "N_SPLITS", "N_REPEATS", "TOP_K", "MIN_RECUR", "FDR_SIG", "FDR_KD")})

## §1 — Inputs: donors, gene set, frozen hyper-parameters, model fit

Everything below was computed by `jobs/run_pseq30.py`. Assertions check the invariants the design relies
on: 30 donors, one gene list for every fit, no NaN in the weight matrix, weight matrix = regulators × 30.

In [ ]:
tab = pd.read_csv(H.DONOR_TAB, index_col=0)
donors = tab.index[tab["retained"]].tolist()
genes = [l.strip() for l in H.GENE_SET.read_text().splitlines() if l.strip()]
hp = H.read_json(H.HYPER)
design = H.load_design()
lic_ord = [l.strip() for l in H.LIC_ORDER.read_text().splitlines() if l.strip()]
react = pd.read_csv(H.REACT_COMP, index_col=0)
annot = H.donor_annotation(tab, react)
cohort = pd.read_csv(H.COHORT_DE, index_col=0)
Zd = pd.read_parquet(H.DONOR_Z)
mf = pd.read_csv(H.MODEL_FIT, index_col=0)
wt = pd.read_csv(H.WEIGHTS, index_col=0)
W = wt[donors]                       # regulators x donors (mean over the 15 folds)
Wsem = pd.read_parquet(H.W_SEM)      # the SEM over folds, same shape
w_cohort = wt["cohort_w"]
w_mean, w_sd = wt["mean_w"], wt["sd_w"]
regs = W.index.tolist()

assert len(donors) == 30 and list(Zd.columns) == donors and list(mf.index[~mf["is_cohort"]]) == donors
assert list(design["genes"]) == genes and Zd.reindex(genes).notna().all().all(), "one gene list for every fit"
assert W.shape == (len(regs), 30) and not W.isna().any().any()
assert set(lic_ord) == set(regs)
sig_genes = cohort.index[cohort["padj"] < H.FDR_SIG].intersection(genes)

print(f"donors: {len(donors)}  (li2024: {(tab.loc[donors, 'study'] == 'li2024').sum()})")
print(tab.loc[donors, ["n_malignant", "n_reactive", "study", "stage", "split"]].to_string())
print(f"\ngene set (Perturb-seq measured x testable in >= {H.GENE_FRAC_DONORS:.0%} of donors): {len(genes):,}")
print(f"cohort30 signature genes at FDR < {H.FDR_SIG}: {len(sig_genes):,}")
print(f"design: {design['Z'].shape[0]:,} genes x {len(regs):,} regulators -> {design['Z'].shape[1]} SVs "
      f"({design['explained'].sum():.1%} of variance)")
print(f"frozen hyper-parameters: alpha = {hp['alpha']:.4g}, l1_ratio = {hp['l1_ratio']:.4g}")

# ---- model fit, sorted by held-out r
fit = mf.loc[donors].sort_values("test_r", ascending=False)
r_cohort = mf.loc["COHORT30", "test_r"]
print(f"\ncohort30 fit: held-out r = {r_cohort:.3f} +- {mf.loc['COHORT30', 'test_r_se']:.3f}  (nb29 cohort: 0.61)")
print(fit[["study", "n_malignant", "n_reactive", "test_r", "test_r_se", "scrambled_r", "train_r",
           "r_vs_consensus_loo", "converged"]].round(3).to_string())
flags = []
flags += [f"{d}: a fold hit max_iter" for d in fit.index[~fit["converged"]]]
flags += [f"{d}: weights correlate {fit.loc[d, 'r_vs_consensus_loo']:.2f} with the leave-one-out consensus (< 0.1)"
          for d in fit.index[fit["r_vs_consensus_loo"] < 0.1]]
print("\nFLAGS:", *flags, sep="\n  ") if flags else print("\nno convergence / consensus flags")

## §2 — F4 Model fit per donor

Held-out Pearson r (mean ± SE over the 15 gene folds) per donor, with the same donor's fit on the shared
scrambled design (each gene's z permuted across regulators) and the 30-donor cohort fit as the ceiling.
Per-donor r is expected well below the cohort's (0.2–0.45); the scrambled points make it interpretable.
Right: r vs min(n cells) — a diagnostic for residual power dependence, not a filter.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7.5, 3), gridspec_kw={"width_ratios": [2.2, 1]})
ax = axs[0]
x = np.arange(len(fit))
ax.errorbar(x, fit["scrambled_r"], yerr=fit["scrambled_r_se"], fmt="o", ms=3, color=SCR, ecolor=SCR,
            elinewidth=0.6, label="scrambled X")
ax.errorbar(x, fit["test_r"], yerr=fit["test_r_se"], fmt="o", ms=3.5, color=TEAL, ecolor=TEAL, elinewidth=0.6,
            label="held-out genes")
ax.axhline(r_cohort, ls="--", lw=0.8, color=DARK)
ax.text(len(fit) - 0.5, r_cohort, f"cohort30 fit r = {r_cohort:.2f}", ha="right", va="bottom", fontsize=6, color=DARK)
ax.axhline(0, lw=0.5, color=MUTED)
ax.set_xticks(x); ax.set_xticklabels([d.replace("Li2024_atlas__", "Li_") for d in fit.index], rotation=90, fontsize=5)
ax.set_ylabel("Pearson r, held-out genes"); ax.legend(frameon=False, loc="center right", fontsize=6)
ax.set_title("Held-out r per donor (mean ± SE over 15 gene folds)", fontsize=7)
ax = axs[1]
rho, p = ss.spearmanr(fit["n_min"], fit["test_r"])
ax.scatter(fit["n_min"], fit["test_r"], s=14, color=TEAL)
ax.set_xscale("log"); ax.set_xlabel("min(n malignant, n reactive)"); ax.set_ylabel("held-out r")
ax.set_title(f"Held-out r vs cell number (Spearman rho = {rho:.2f}, p = {p:.2g})", fontsize=7)
med, scr_med = fit["test_r"].median(), fit["scrambled_r"].median()
fig.suptitle(f"Per-donor model fit: held-out r (teal) vs scrambled-X r (grey); median {med:.2f} vs {scr_med:.2f}, cohort30 {r_cohort:.2f}",
             fontsize=8)
H.savefig(fig, "model_fit")
print(f"Spearman rho(r, min n cells) = {rho:.3f}, p = {p:.3g}")

## §3 — Regulator weights per donor: the Long-Island-City plots and recurrence

Every LIC panel uses the **same x ordering** (regulators clustered on their knockdown effects over the
cohort30 signature genes; average linkage, correlation distance), so the 30 panels and the consensus are
aligned. **3.1 Recurrence:** per donor the top 25 and bottom 25 regulators by $w$; per regulator, in how
many donors it sits in the top set (consistent positive sign) or the bottom set (consistent negative sign).
Chance expectation ≈ 30 × 50 / 2,582 ≈ 0.6 donors per regulator; a binomial p is given for completeness.

In [ ]:
rec = H.recurrence(W, top_k=H.TOP_K, min_recur=H.MIN_RECUR)
rec["cluster"] = P.regulator_cluster_map(P.load_clusters()).reindex(rec.index)
rec.to_csv(H.RECURRENCE)
recurrent = rec.index[rec["recurrent"]].tolist()
print(f"expected recurrences per regulator under chance: {rec.attrs['expected_chance']:.2f}")
print(f"recurrent regulators (top/bottom {H.TOP_K} in >= {H.MIN_RECUR}/30 donors): {len(recurrent)}  "
      f"(positive {int((rec.loc[recurrent, 'direction'] == 'positive').sum())}, "
      f"negative {int((rec.loc[recurrent, 'direction'] == 'negative').sum())})")
print(rec.head(30)[["n_top", "n_bottom", "n_recur", "direction", "mean_w", "sd_w", "frac_same_sign", "q_binom", "cluster"]]
      .round(3).to_string())

In [ ]:
# F1 -- 30 LIC panels, 5 x 6, shared x ordering and y limits; teal = that donor's top/bottom 25; 3 + 3 labels
ylim = (float(W.min().min()) * 1.05, float(W.max().max()) * 1.05)
fig, axs = plt.subplots(5, 6, figsize=(12, 8.5), sharex=True, sharey=True)
for ax, d in zip(axs.ravel(), fit.index):
    w = W[d]
    hi = w.nlargest(H.TOP_K).index.tolist() + w.nsmallest(H.TOP_K).index.tolist()
    lab = w.nlargest(3).index.tolist() + w.nsmallest(3).index.tolist()
    H.lic_panel(ax, w, lic_ord, highlight=hi, labels=lab, ylim=ylim, s=1.5, label_fs=4.5)
    ax.set_title(f"{d.replace('Li2024_atlas__', 'Li_')} · {tab.loc[d, 'n_malignant']:,}/{tab.loc[d, 'n_reactive']:,} cells · "
                 f"r = {fit.loc[d, 'test_r']:.2f}", fontsize=6, color=DARK)
    ax.tick_params(labelsize=5)
for ax in axs[-1]:
    ax.set_xlabel("regulators (shared LIC order)", fontsize=6)
for ax in axs[:, 0]:
    ax.set_ylabel("$w$", fontsize=7)
fig.suptitle(f"Regulator weights per donor (donors sorted by held-out r; teal = that donor's top/bottom {H.TOP_K} regulators)",
             fontsize=9, y=1.0)
fig.tight_layout()
H.savefig(fig, "lic_per_donor")

In [ ]:
# F2 -- consensus LIC: mean w across the 30 donors (± SD); colour = recurrence tier, i.e. the fraction of donors in
# whose top/bottom 25 the regulator sits (> 50 %, > 40 %, > 30 %); every coloured regulator is named
TIERS = [("> 50 %", 0.5, TEAL), ("> 40 %", 0.4, "#D97706"), ("> 30 %", 0.3, "#7C3AED")]
frac_recur = rec["n_recur"] / len(donors)
tier_of = {}
for name, thr, col in TIERS[::-1]:              # lowest tier first so a higher tier overwrites
    for r in rec.index[frac_recur > thr]:
        tier_of[r] = (name, col)
tiered = [r for r in rec.sort_values("n_recur", ascending=False).index if r in tier_of]
tier_colors = {r: tier_of[r][1] for r in tiered}
fig, ax = plt.subplots(figsize=(9.5, 4.2))
H.lic_panel(ax, w_mean, lic_ord, highlight=tiered, labels=tiered, colors=tier_colors, yerr=w_sd, s=2.5, label_fs=5)
for name, thr, col in TIERS:
    n = sum(1 for r in tiered if tier_of[r][0] == name)
    ax.scatter([], [], color=col, s=14, label=f"in top/bottom {H.TOP_K} of {name} of donors (> {int(thr * len(donors))}/30; n = {n})")
ax.legend(frameon=False, fontsize=6, loc="upper left")
ax.set_ylabel("mean $w$ across 30 donors (± SD)")
ax.set_xlabel(f"{len(regs):,} regulators, clustered on their knockdown effects on {len(sig_genes):,} cohort signature genes")
ax.set_title("Mean regulator weight across donors; colour = share of donors with the regulator in their top/bottom 25", fontsize=8)
H.savefig(fig, "lic_consensus_recurrent")

In [ ]:
# the recurrence tiers coloured in F2, printed (each regulator listed once, in its highest tier)
cols = ["n_top", "n_bottom", "n_recur", "direction", "mean_w", "sd_w", "frac_same_sign", "q_binom", "cluster"]
for name, thr, _ in TIERS:
    members = [r for r in tiered if tier_of[r][0] == name]
    n_pos = sum(rec.loc[r, "direction"] == "positive" for r in members)
    print(f"\n=== recurrent in {name} of donors (> {int(thr * len(donors))}/30): {len(members)} regulators "
          f"({n_pos} w > 0 'knockdown mimics MF', {len(members) - n_pos} w < 0 'MF depends on it') ===")
    print(rec.loc[members, cols].round(3).to_string() if members else "none")

In [ ]:
# F2b -- the same consensus LIC with only the > 50 % tier highlighted (the most recurrent regulators)
top50 = [r for r in tiered if tier_of[r][0] == TIERS[0][0]]
fig, ax = plt.subplots(figsize=(9.5, 3.8))
H.lic_panel(ax, w_mean, lic_ord, highlight=top50, labels=[], yerr=w_sd, s=2.5)
pos = {r: i for i, r in enumerate(lic_ord)}
for r in top50:
    ax.annotate(f"{r}\n{rec.loc[r, 'direction']} in {int(rec.loc[r, 'n_recur'])}/30 donors",
                (pos[r], w_mean[r]), fontsize=7, fontweight="bold", color=DARK, xytext=(6, -4), textcoords="offset points")
ax.set_ylabel("mean $w$ across 30 donors (± SD)")
ax.set_xlabel(f"{len(regs):,} regulators, clustered on their knockdown effects on {len(sig_genes):,} cohort signature genes")
ax.set_title(f"Mean regulator weight across donors; teal = regulators in the top/bottom {H.TOP_K} of > 50 % of donors "
             f"({', '.join(top50)})", fontsize=8)
H.savefig(fig, "lic_consensus_top50")

# per-donor view of the same regulators: weight and rank in each donor, counted from the regulator's own direction
ranks_neg = W.rank(axis=0, ascending=True).astype(int)
ranks_pos = W.rank(axis=0, ascending=False).astype(int)
view = pd.DataFrame({f"{r} w": W.loc[r].round(4) for r in top50})
for r in top50:
    view[f"{r} rank"] = (ranks_neg if rec.loc[r, "direction"] == "negative" else ranks_pos).loc[r]
view["stage"] = annot["stage"].reindex(view.index)
print(f"rank = position of the regulator in that donor's weights, counted from the regulator's own direction (1 = strongest of {len(regs):,})")
print(view.sort_values(f"{top50[0]} rank").to_string())

## §4 — 3.2 Do patients group by their regulator weights? (F5)

Spearman correlation between donors across all regulator weights → distance 1 − ρ → Ward linkage; k over
2–5 by mean silhouette (profile printed so the choice is visible). Cluster membership is tested against study,
stage, cell number, CNV burden and the **reactive-pool composition control** (if donors group by the
CD8A / FOXP3 / cytotoxic content of their reactive CD4 pool, the clustering reads the reference, not the
tumour). li2024 contributes 14/30 donors and uses Flex chemistry — if clusters align with study, that is
stated plainly.

In [ ]:
K_FIXED = 4  # clustering resolution: fixed at 4 clusters (silhouette optimum is printed for reference)
cl = H.donor_clustering(W, k_range=H.K_RANGE)
print("silhouette by k:", cl["silhouette"].round(3).to_dict(), f"-> silhouette optimum k = {cl['k']}; using k = {K_FIXED}")
cl["k"] = K_FIXED
cl["labels"] = pd.Series(sch.fcluster(cl["link"], K_FIXED, criterion="maxclust"), index=cl["rho"].index, name="cluster")
labels = cl["labels"]
print("cluster sizes:", labels.value_counts().sort_index().to_dict())
for k in sorted(labels.unique()):
    print(f"  cluster {k}: " + ", ".join(d.replace("Li2024_atlas__", "Li_") for d in labels.index[labels == k]))
cov = H.cluster_covariate_tests(labels, annot)
print("\ncluster vs covariates:")
print(cov.round(4).to_string(index=False))
ct_study = pd.crosstab(annot.loc[labels.index, "study"], labels)
print("\nstudy x cluster:"); print(ct_study.to_string())
print("\nstage x cluster:"); print(pd.crosstab(annot.loc[labels.index, "stage"], labels).to_string())
p_study = float(cov.loc[cov["covariate"] == "study", "p"].iloc[0])
p_react = cov.loc[cov["covariate"].str.startswith("react_"), "p"].min()
print("\nVERDICT:",
      "clusters ALIGN WITH STUDY (p < 0.05) -- treat them as batch, not biology." if p_study < 0.05 else
      "clusters do not track study (p >= 0.05).",
      "Reactive-pool composition differs between clusters (p < 0.05) -- the reference may drive them." if p_react < 0.05 else
      "Reactive-pool composition does not differ between clusters.")
out = annot.loc[labels.index].copy(); out.insert(0, "cluster", labels); out["test_r"] = fit["test_r"].reindex(out.index)
out.to_csv(H.CLUSTERS)

# ---- F5: dendrogram + donor x donor Spearman rho + covariate bars
leaves = cl["leaves"]
rho = cl["rho"].loc[leaves, leaves]
bars = [("cluster", labels, "cat"), ("study", annot["study"], "cat"), ("stage", annot["stage"], "cat"),
        ("log10 n cells", np.log10(annot["n_min"]), "num")]
fig = plt.figure(figsize=(6.2, 6.6))
gs = fig.add_gridspec(2 + len(bars), 2, height_ratios=[0.9, 5] + [0.22] * len(bars), width_ratios=[5, 0.25],
                      hspace=0.06, wspace=0.04)
# cluster colours shared by the dendrogram branches, the heatmap block labels, the tick labels and the cluster bar
ccol = {k: plt.get_cmap("Set2")(j) for j, k in enumerate(sorted(labels.unique()))}
n_d = len(labels)
node_cl = {i: {int(labels.iloc[i])} for i in range(n_d)}
for j, (a, b, *_rest) in enumerate(cl["link"]):
    node_cl[n_d + j] = node_cl[int(a)] | node_cl[int(b)]
axd = fig.add_subplot(gs[0, 0])
with plt.rc_context({"lines.linewidth": 1.0}):
    sch.dendrogram(cl["link"], ax=axd, no_labels=True,
                   link_color_func=lambda i: mpl.colors.to_hex(ccol[next(iter(node_cl[i]))]) if len(node_cl[i]) == 1 else MUTED)
axd.axis("off")
axh = fig.add_subplot(gs[1, 0])
im = axh.imshow(rho.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto", interpolation="nearest")
axh.set_xticks([]); axh.set_yticks(range(len(leaves)))
axh.set_yticklabels([d.replace("Li2024_atlas__", "Li_") for d in leaves], fontsize=5)
lv = labels.reindex(leaves).to_numpy()
for t, k in zip(axh.get_yticklabels(), lv):
    t.set_color(ccol[int(k)]); t.set_fontweight("bold")
# white separators between clusters + a label in the middle of each diagonal block
bounds = [0] + [i for i in range(1, len(lv)) if lv[i] != lv[i - 1]] + [len(lv)]
for b in bounds[1:-1]:
    axh.axhline(b - 0.5, color="white", lw=1.5); axh.axvline(b - 0.5, color="white", lw=1.5)
for s0, s1 in zip(bounds[:-1], bounds[1:]):
    k = int(lv[s0]); c = (s0 + s1 - 1) / 2
    axh.text(c, c, f"cluster {k}\n(n = {s1 - s0})", ha="center", va="center", fontsize=6.5, fontweight="bold", color=DARK,
             bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=ccol[k], lw=1.2, alpha=0.95), zorder=5)
cax = fig.add_subplot(gs[1, 1]); fig.colorbar(im, cax=cax).set_label("Spearman rho of $w$", fontsize=6); cax.tick_params(labelsize=5)
cat_cmaps = {"cluster": plt.get_cmap("Set2"), "study": plt.get_cmap("tab10"), "stage": plt.get_cmap("Pastel1")}
for i, (name, v, kind) in enumerate(bars):
    axb = fig.add_subplot(gs[2 + i, 0])
    v = v.reindex(leaves)
    if kind == "cat":
        cats = sorted(v.dropna().astype(str).unique())
        cmap = cat_cmaps[name]
        col = {c: cmap(j % cmap.N) for j, c in enumerate(cats)}
        axb.imshow([[cats.index(str(x)) if pd.notna(x) else -1 for x in v]], cmap=mpl.colors.ListedColormap([col[c] for c in cats]),
                   aspect="auto", vmin=0, vmax=max(len(cats) - 1, 1), interpolation="nearest")
        for j, c in enumerate(cats):
            axb.scatter([], [], marker="s", color=col[c], label=f"{name}={c}", s=12)
    else:
        axb.imshow([v.to_numpy(dtype=float)], cmap="Greys", aspect="auto", interpolation="nearest")
    axb.set_xticks([]); axb.set_yticks([0]); axb.set_yticklabels([name], fontsize=5.5)
    for s in axb.spines.values():
        s.set_visible(False)
handles = [h for ax in fig.axes for h in ax.get_legend_handles_labels()[0]]
lbls = [l for ax in fig.axes for l in ax.get_legend_handles_labels()[1]]
fig.legend(handles, lbls, loc="center left", bbox_to_anchor=(1.0, 0.3), frameon=False, fontsize=5, ncol=1)
fig.suptitle(f"Donor × donor Spearman correlation of regulator weights; Ward dendrogram cut at k = {K_FIXED} clusters "
             f"(silhouette {cl['silhouette'][K_FIXED]:.2f})", fontsize=8, y=0.92)
H.savefig(fig, "donor_clustering")


In [ ]:
# per cluster: mean w over its donors; top 10 most negative (MF depends on the regulator in these donors) and
# top 10 most positive (knockdown mimics MF in these donors). q = one-vs-rest Mann-Whitney across donors, BH over regulators.
from statsmodels.stats.multitest import multipletests
reg_cluster = P.regulator_cluster_map(P.load_clusters())
N_SHOW = 10
rows = []
for k in sorted(labels.unique()):
    inside, outside = labels.index[labels == k], labels.index[labels != k]
    d = pd.DataFrame({"regulator": W.index, "cluster_id": k,
                      "mean_w_in": W[inside].mean(axis=1).to_numpy(), "mean_w_out": W[outside].mean(axis=1).to_numpy()})
    d["diff_in_minus_out"] = d["mean_w_in"] - d["mean_w_out"]
    d["p"] = [ss.mannwhitneyu(W.loc[r, inside], W.loc[r, outside], alternative="two-sided").pvalue for r in W.index]
    d["q"] = multipletests(d["p"], method="fdr_bh")[1]
    d["authors_cluster"] = reg_cluster.reindex(d["regulator"]).to_numpy()
    d["n_recur"] = rec["n_recur"].reindex(d["regulator"]).to_numpy()
    rows.append(d)
cm = pd.concat(rows, ignore_index=True).sort_values(["cluster_id", "mean_w_in"])
cm.to_csv(H.table("cluster_markers"), index=False)
cols = ["regulator", "mean_w_in", "mean_w_out", "q", "n_recur", "authors_cluster"]
for k, sub in cm.groupby("cluster_id"):
    members = [d.replace("Li2024_atlas__", "Li_") for d in labels.index[labels == k]]
    stage_in = annot.loc[labels.index[labels == k], "stage"].value_counts().to_dict()
    print(f"\n=== cluster {k}: n = {len(members)} donors, stage {stage_in}, {int((sub['q'] < 0.1).sum())} regulators differ from the rest at q < 0.1 ===")
    print("    " + ", ".join(members))
    print(f"\n-- top {N_SHOW} NEGATIVE regulators in cluster {k} (MF program depends on them) --")
    print(sub.sort_values("mean_w_in").head(N_SHOW)[cols].round(4).to_string(index=False))
    print(f"\n-- top {N_SHOW} POSITIVE regulators in cluster {k} (knockdown mimics the MF program) --")
    print(sub.sort_values("mean_w_in", ascending=False).head(N_SHOW)[cols].round(4).to_string(index=False))


## §5 — 3.3 Programs: which regulator sets carry the consensus and the cluster-specific weights? (F3)

nb29's signed pathway tests over the authors' 111 functional clusters and the project's TCR/costim modules
(≥ 5 members): one-sided Mann–Whitney for members' $w$ > rest ("knockdown mimics the state") and members'
$w$ < rest ("the state depends on the set"), BH across sets. Run for the consensus (mean $w$) and for each
donor cluster's mean $w$. Programs significant in only one cluster are flagged, not interpreted.

In [ ]:
clusters_df = P.load_clusters()
sets = {**P.cluster_sets(clusters_df), **{f"TCRmod:{k}": v for k, v in TH.TCR_MODULES.items()}}
w_groups = {"consensus": w_mean, **{f"cluster{k}": W[labels.index[labels == k]].mean(axis=1) for k in sorted(labels.unique())}}
prog = H.program_tests(w_groups, sets, min_n=5)
prog.to_csv(H.PROGRAMS, index=False)
sig_prog = prog[prog["q_signed"] < 0.1]
print(f"significant programs (q < 0.1): {len(sig_prog)} rows over {sig_prog['set'].nunique()} sets")
for g, sub in sig_prog.groupby("group"):
    print(f"\n[{g}]")
    print(sub[["set", "n", "mean_w", "delta_signed", "direction", "q_signed"]].round(3).to_string(index=False))
non = sig_prog[sig_prog["group"] != "consensus"]
cons_sets = set(sig_prog.loc[sig_prog["group"] == "consensus", "set"])
only_one = [s for s, n in non.groupby("set")["group"].nunique().items() if n == 1 and s not in cons_sets]
print("\nFLAG -- significant in exactly one cluster and not in the consensus:", only_one if only_one else "none")

# ---- F3: consensus LIC coloured by the significant consensus programs (top 8 by q)
cons_sig = sig_prog[sig_prog["group"] == "consensus"].sort_values("q_signed").head(8)
pal = plt.get_cmap("Dark2")
colors, members_all = {}, []
for j, (_, row) in enumerate(cons_sig.iterrows()):
    mem = [m for m in sets[row["set"]] if m in W.index]
    for m in mem:
        colors.setdefault(m, pal(j))
    members_all += mem
fig, ax = plt.subplots(figsize=(7, 3.2))
H.lic_panel(ax, w_mean, lic_ord, highlight=list(dict.fromkeys(members_all)), colors=colors, s=2.5)
for j, (_, row) in enumerate(cons_sig.iterrows()):
    ax.scatter([], [], color=pal(j), s=12, label=f"{row['set'][:38]} (n={row['n']}, q={row['q_signed']:.2g}, {row['direction'].split(' (')[0]})")
ax.legend(frameon=False, fontsize=5, loc="upper left", bbox_to_anchor=(1.0, 1.0))
ax.set_ylabel("mean $w$ across 30 donors"); ax.set_xlabel("regulators (shared LIC order)")
ax.set_title(f"Mean regulator weight across donors; colour = members of the {len(cons_sig)} programs significant for the consensus (q < 0.1)"
             if len(cons_sig) else "Mean regulator weight across donors; no program significant for the consensus (q < 0.1)", fontsize=8)
H.savefig(fig, "lic_consensus_programs")

## §6 — 3.4 Downstream of the top recurrent regulators (F6)

Top 2–3 regulators by recurrence (both signs if both recur). The authors' rule, verbatim: a cohort30
signature gene (FDR < 0.01) is **controlled** by regulator $k$ if (a) it is DE at 10 % FDR on knockdown of
$k$ and (b) the sign of its knockdown z is consistent with both the sign of $w_k$ and the sign of the
signature ($w > 0$: knockdown-up genes are up in malignant; $w < 0$: knockdown-up genes are down in
malignant). Reads two layers of the Perturb-seq h5ad (~1 GB).

In [ ]:
pos = rec[rec["recurrent"] & (rec["direction"] == "positive")].head(2).index.tolist()
neg = rec[rec["recurrent"] & (rec["direction"] == "negative")].head(2).index.tolist()
top_regs = (pos[:2] + neg[:1]) if (pos and neg) else (pos + neg)[:3]
if not top_regs:
    top_regs = rec.head(3).index.tolist(); print("no regulator passes the recurrence cutoff; using the top 3 by n_recur")
print("top regulators:", {r: f"{rec.loc[r, 'direction']} n_recur={int(rec.loc[r, 'n_recur'])} mean_w={rec.loc[r, 'mean_w']:+.3f}" for r in top_regs})

pseq = P.load_pseq(H.CONDITION, H.PSEQ_H5, layers=("zscore", "adj_p_value"))
ds_sets = H.downstream_sets(H.CONDITION)
universe = set(sig_genes)
ctrl, enr, top_side = [], [], {}
N_LABEL = 8
for r in top_regs:
    df = H.controlled_genes(pseq, r, float(w_mean[r]), cohort, fdr_sig=H.FDR_SIG, fdr_kd=H.FDR_KD)
    df["n_recur"] = int(rec.loc[r, "n_recur"]); ctrl.append(df)
    c = df[df["controlled"]]
    up = c[c["lfc_cohort"] > 0].sort_values("padj_cohort").head(N_LABEL)
    dn = c[c["lfc_cohort"] < 0].sort_values("padj_cohort").head(N_LABEL)
    top_side[r] = (up, dn)
    print(f"\n{r} (w = {w_mean[r]:+.3f}, recurs in {int(rec.loc[r, 'n_recur'])}/30): {len(df)} signature genes DE on knockdown, "
          f"{len(c)} controlled ({len(c) / max(len(df), 1):.0%} sign-consistent); "
          f"{int((c['lfc_cohort'] > 0).sum())} up in malignant, {int((c['lfc_cohort'] < 0).sum())} down in malignant")
    print(f"  most significant controlled genes UP in malignant:   " + ", ".join(up.index))
    print(f"  most significant controlled genes DOWN in malignant: " + ", ".join(dn.index))
    print("  top 15 controlled genes by |knockdown z|:")
    print(c.head(15)[["z_kd", "lfc_cohort", "padj_cohort"]].round(3).to_string())
    e = H.hypergeom_enrichment(c.index, universe, ds_sets)
    if len(e):
        e.insert(0, "regulator", r); enr.append(e)
        print(f"  enrichment over the authors' downstream programs (q < 0.1): "
              f"{e[e['q'] < 0.1][['set', 'overlap', 'expected', 'q']].round(3).to_string(index=False) if (e['q'] < 0.1).any() else 'none'}")
ctrl = pd.concat(ctrl)
ctrl.to_csv(H.DOWNSTREAM)
enr = pd.concat(enr, ignore_index=True) if enr else pd.DataFrame()

# ---- F6: one volcano per regulator on the cohort30 signature; teal = genes controlled by the regulator;
#          bold labels = the N_LABEL most significant controlled genes on each side (up / down in malignant)
fig, axs = plt.subplots(1, len(top_regs), figsize=(4.8 * len(top_regs) + 0.6, 4.8), sharey=True, squeeze=False)
vol = cohort.reindex(genes).dropna(subset=["padj"])
for ax, r in zip(axs.ravel(), top_regs):
    df = ctrl[ctrl["regulator"] == r]
    c = df[df["controlled"]]
    up, dn = top_side[r]
    ax.scatter(vol["log2FoldChange"], -np.log10(vol["padj"].clip(1e-300)), s=2, color=GREY, lw=0, rasterized=True,
               label="cohort30 signature gene")
    ax.scatter(c["lfc_cohort"], -np.log10(c["padj_cohort"].clip(1e-300)), s=9, color=TEAL, lw=0, zorder=3,
               label=f"controlled by {r} ({len(c)})")
    for g, row in pd.concat([up, dn]).iterrows():
        ax.annotate(g, (row["lfc_cohort"], -np.log10(max(row["padj_cohort"], 1e-300))), fontsize=6.5, fontweight="bold",
                    xytext=(3, 2), textcoords="offset points", color=DARK, zorder=4)
    ax.axvline(0, lw=0.5, color=MUTED, ls="--")
    ax.set_title(f"{r}: w = {w_mean[r]:+.3f}, in the bottom/top {H.TOP_K} of {int(rec.loc[r, 'n_recur'])}/30 donors\n"
                 f"{len(c)} of {len(df)} knockdown-DE signature genes are sign-consistent (controlled)", fontsize=7.5)
    ax.set_xlabel("cohort30 log2FC (malignant / reactive)")
    ax.legend(frameon=False, fontsize=6, loc="upper right")
axs[0, 0].set_ylabel("-log10 FDR (cohort30 malignant vs reactive)")
fig.suptitle("Cohort30 signature volcano per recurrent regulator; teal = genes the regulator controls; bold = most significant per side",
             fontsize=8.5)
fig.tight_layout()
H.savefig(fig, "downstream")
del pseq

## §7 — Full numerical report

Every number above, re-printed in one block and saved to `tables/pseq30_full_report_v1.txt`. Each section is
guarded, so a partial run prints the error for that block and continues.

In [ ]:
import io as _io
_buf = _io.StringIO()
def _p(*a, **k):
    print(*a, **k); print(*a, **k, file=_buf)
def _hdr(t): _p("\n" + "=" * 100 + f"\n{t}\n" + "=" * 100)
def _sub(t): _p("\n" + "-" * 100 + f"\n{t}\n" + "-" * 100)
def _sec(title, fn):
    _sub(title)
    try: fn()
    except Exception as e: _p(f"[not available in this run: {type(e).__name__}: {e}]")

_hdr("30_perturbseq_per_donor -- FULL REPORT")
_p(f"""
WHAT WAS DONE
Per-donor pert2state on the GWCD4i {H.CONDITION} knockdown resource. Cohort: nb29's MF CD4-clone skin donors with
>= {H.MIN_CELLS_ARM} malignant and >= {H.MIN_CELLS_ARM} reactive CD4 cells (30 donors). Per donor: {H.N_PSEUDO} + {H.N_PSEUDO} seeded pseudo-replicate
pseudobulks -> pydeseq2 ~ arm -> Wald stat = y_donor. Cohort reference: pydeseq2 ~ donor + arm on the same 30 donors.
Design built ONCE: {len(genes):,} genes (measured in Perturb-seq x testable in >= {H.GENE_FRAC_DONORS:.0%} donors) x {len(regs):,}
regulators (zscore_masked), column-standardized, TruncatedSVD D = {H.N_PCS}. Hyper-parameters frozen once on the cohort
signature (ElasticNetCV, 4-fold): alpha = {hp['alpha']:.4g}, l1_ratio = {hp['l1_ratio']:.4g}. Every fit = ElasticNet on the
same 60-dim projection with the same {H.N_SPLITS}-fold x {H.N_REPEATS} gene folds; w = V^T coef averaged over folds. Null =
the same donor fitted on one shared scrambled design (each gene's z permuted across regulators).
Sign: w > 0 knockdown mimics the malignant program; w < 0 the program depends on the regulator.
""")
_sec("1. Retained donors", lambda: _p(tab.loc[donors, ["n_malignant", "n_reactive", "study", "stage", "split"]].to_string()))
_sec("2. Model fit (sorted by held-out r)", lambda: (_p(f"cohort30 r = {r_cohort:.3f}"),
     _p(fit[["study", "n_min", "test_r", "test_r_se", "scrambled_r", "train_r", "r_vs_consensus_loo", "converged"]].round(3).to_string()),
     _p(f"Spearman rho(r, min n cells) = {rho:.3f} p = {p:.3g}"), _p("flags: " + ("; ".join(flags) if flags else "none"))))
_sec("3. Recurrence (top 30)", lambda: (_p(f"chance expectation {rec.attrs['expected_chance']:.2f}; recurrent (>= {H.MIN_RECUR}/30): {len(recurrent)}"),
     _p(rec.head(30)[["n_top", "n_bottom", "n_recur", "direction", "mean_w", "sd_w", "frac_same_sign", "q_binom", "cluster"]].round(3).to_string())))
_sec("4. Donor clustering", lambda: (_p("silhouette by k: " + str(cl["silhouette"].round(3).to_dict()) + f"; k = {cl['k']}"),
     _p(out[["cluster", "study", "stage", "n_min", "n_event_arms", "react_CD8A_frac", "react_FOXP3_frac", "test_r"]].round(3).to_string()),
     _p(cov.round(4).to_string(index=False)), _p(ct_study.to_string())))
_sec("5. Programs (q < 0.1)", lambda: (_p(sig_prog[["group", "set", "n", "mean_w", "delta_signed", "direction", "q_signed"]].round(3).to_string(index=False)),
     _p("significant in exactly one cluster: " + (", ".join(only_one) if only_one else "none"))))
_sec("6. Downstream of the top recurrent regulators", lambda: (
     [_p(f"{r}: {int((ctrl['regulator'] == r).sum())} KD-DE signature genes, {int(ctrl.loc[ctrl['regulator'] == r, 'controlled'].sum())} controlled; "
         f"top 15: " + ", ".join(ctrl[(ctrl['regulator'] == r) & ctrl['controlled']].head(15).index)) for r in top_regs],
     _p(enr[enr["q"] < 0.1][["regulator", "set", "overlap", "expected", "q"]].round(3).to_string(index=False) if len(enr) else "no enrichment rows")))
_sec("7. Outputs", lambda: _p("\n".join(str(p_) for p_ in (H.MODEL_FIT, H.WEIGHTS, H.RECURRENCE, H.CLUSTERS, H.PROGRAMS, H.DOWNSTREAM, H.REPORT))
     + "\nfigures: " + ", ".join(f"pseq30_{n}" for n in ("lic_per_donor", "lic_consensus_recurrent", "lic_consensus_programs", "model_fit", "donor_clustering", "downstream"))))
H.REPORT.write_text(_buf.getvalue())
print("\nsaved", H.REPORT)